# 2. Evaluating the agents against ground truth

The honest state of play: **the agents have never been evaluated on real
incidents.** What we know from the team's earlier vision research (the
notebook itself, `research/vision_confidence/`, was removed from this repo
for disk space -- these numbers are what remains of it):

- Best prompt strategy reached about **54% top-1 species accuracy**.
- **50 of 485 answers given at 90%+ confidence were wrong.**
- The strategy comparison used **5 photos**, one of which was a dog control.
- The injury and distress flags have **never been evaluated at all**.

That last point matters most, because those flags feed severity scoring,
which decides whether a professional team is sent.

## Where ground truth comes from

The `incident_logs` table. When a responder closes an incident they record
`confirmed_species` and `report_was_accurate` -- see
[`LogIncidentScreen.tsx`](../clients/responder_console/src/screens/LogIncidentScreen.tsx).
That is the only ground truth the system ever receives.

This notebook is the harness, ready for when that data exists.

In [ ]:
# Auto-apply backend code edits without restarting the kernel: after you
# (or Claude) edit a .py file under backend/lifejacket/, the next cell you
# run picks up the change automatically -- no "Restart Kernel" needed.
#
# This does NOT cover prompts/*.md (those are cached by load_prompt(), not
# reloaded as Python modules) -- call clear_prompt_cache() for those, or use
# the cell further down that already does this after a prompt edit.
%load_ext autoreload
%autoreload 2

# Make the backend package importable from the notebooks/ directory.
import sys, pathlib

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'backend'))

print('repo root:', REPO_ROOT)

In [ ]:
import pandas as pd
from sqlalchemy import select

from lifejacket.models.db import SessionLocal, init_db
from lifejacket.models.tables import IncidentLogRow, IncidentRow

init_db()
session = SessionLocal()

## Pairing predictions with outcomes

An inner join on `incident_id`: every closed incident that has both an
agent prediction and a responder's correction.

In [ ]:
rows = session.execute(
    select(
        IncidentRow.incident_id,
        IncidentRow.species_common_name.label('predicted_species'),
        IncidentRow.species_confidence.label('predicted_confidence'),
        IncidentRow.animal_group.label('predicted_group'),
        IncidentRow.severity_level,
        IncidentRow.severity_score,
        IncidentRow.injury_present,
        IncidentRow.entanglement_present,
        IncidentLogRow.confirmed_species,
        IncidentLogRow.confirmed_animal_group,
        IncidentLogRow.outcome,
        IncidentLogRow.report_was_accurate,
    ).join(IncidentLogRow, IncidentRow.incident_id == IncidentLogRow.incident_id)
).all()

evaluation = pd.DataFrame(rows)
print(f'{len(evaluation)} closed incidents with ground truth')
evaluation.head(20)

## Metric 1: species accuracy

Compared on normalised names, so "Harbor Seal" and "harbour seal" match.

In [ ]:
from lifejacket.dispatch.species import group_from_species, normalise_species


def build_species_frame(evaluation):
    """Add correctness columns. Returns an empty frame when there is no truth yet."""
    if evaluation.empty:
        return evaluation
    df = evaluation.dropna(subset=['confirmed_species']).copy()
    df['species_correct'] = df.apply(
        lambda r: normalise_species(r.predicted_species) == normalise_species(r.confirmed_species),
        axis=1,
    )
    # Group accuracy is the more operationally meaningful number: routing is
    # by group, so a harbour-porpoise-for-common-dolphin error still reaches
    # the right team.
    df['group_correct'] = df.apply(
        lambda r: group_from_species(r.predicted_species) == group_from_species(r.confirmed_species),
        axis=1,
    )
    return df


species = build_species_frame(evaluation)

if species.empty:
    print('No ground truth yet. Close some incidents with logs first.')
else:
    print(f'species top-1 : {species.species_correct.mean():.1%}  (n={len(species)})')
    print(f'animal group  : {species.group_correct.mean():.1%}')

## Metric 2: is the 0.90 confidence threshold safe?

**This is the most important question in the project.**

`SPECIES_CONFIDENCE_THRESHOLD = 0.90` is where the identification agent
stops asking questions and commits. The research data says roughly 10% of
90%+ confident answers were wrong. If that holds on real incidents, the
threshold is too low and should move.

A well-calibrated model would put accuracy within each confidence bucket
close to the bucket's own confidence.

In [ ]:
if species.empty:
    print('Needs ground truth data.')
else:
    buckets = pd.cut(species.predicted_confidence, [0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0])
    calibration = species.groupby(buckets, observed=True).agg(
        n=('species_correct', 'size'),
        accuracy=('species_correct', 'mean'),
        mean_confidence=('predicted_confidence', 'mean'),
    )
    calibration['overconfidence'] = calibration.mean_confidence - calibration.accuracy
    display(calibration)
    print()
    print('A positive overconfidence value means the agent is more sure than it')
    print('deserves to be in that bucket. Watch the top two rows especially.')

## Metric 3: severity vs. outcome

Two error types, with very different costs:

- **Under-triage**: scored `guidance` or `monitor`, but the animal needed
  rescue. This is the dangerous direction.
- **Over-triage**: scored `critical`, but no action was needed. Wastes a
  team and erodes trust in the alerts.

Under-triage should be weighted far more heavily when tuning
`config/scoring.json`.

In [ ]:
if evaluation.empty:
    print('Needs ground truth data.')
else:
    needed_rescue = {'rescued', 'released', 'deceased'}
    sev = evaluation.dropna(subset=['outcome', 'severity_level']).copy()
    sev['needed_response'] = sev.outcome.isin(needed_rescue)
    sev['was_low_triage'] = sev.severity_level.isin(['guidance', 'monitor'])

    print(pd.crosstab(sev.severity_level, sev.outcome))
    print()
    under = sev[sev.needed_response & sev.was_low_triage]
    over = sev[~sev.needed_response & (sev.severity_level == 'critical')]
    print(f'UNDER-triaged (dangerous): {len(under)} of {len(sev)}')
    print(f'OVER-triaged  (wasteful) : {len(over)} of {len(sev)}')
    if len(under):
        display(under[['incident_id', 'predicted_species', 'confirmed_species', 'severity_level', 'outcome']])

## Metric 4: did responders find the report accurate?

The most direct signal on the report agent, and the cheapest to collect.

In [ ]:
if evaluation.empty:
    print('Needs ground truth data.')
else:
    accuracy = evaluation.report_was_accurate.dropna()
    if len(accuracy):
        print(f'reports judged accurate: {accuracy.mean():.1%}  (n={len(accuracy)})')
    else:
        print('No responder has rated a report yet.')

## Backlog

In rough priority order:

1. **Injury flag accuracy.** Never measured, and it feeds severity. Needs a
   labelled photo set -- the 400-image hurt-animal dataset from the earlier
   vision research is gone along with that folder; start a fresh one.
2. **Re-run the vision strategy comparison on more than 5 photos.** The
   current conclusion rests on almost nothing.
3. **Calibrate the 0.90 threshold** against the table above.
4. **Question efficiency.** How many of the five questions are actually
   needed before confidence stops moving? Transcripts are in `chat_messages`.
5. **Guardrail false-positive rate.** How often is a report held for review
   that a human then approves unchanged? Too high and coordinators will
   stop reading the warnings.
6. **Dispatch accuracy.** The directory-vs-online-search benchmark in
   `research/dispatch_benchmark/` was built but never run; both of its run
   switches are still `False`.

In [ ]:
session.close()